# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIRˆ2) Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIRˆ2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'
# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")
# Optionally display further info, e.g. version, date published
print(f"\nVersion: {getattr(metadata, 'version', 'N/A')}")
print(f"Date Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll use `dataset.record_sets` to explore all record sets and their metadata, referencing each by its Croissant `@id`. Each record set may contain fields/columns, also indexed by `@id`.

Let's enumerate the main record sets and their available fields.

In [ ]:
# Discover all record sets defined in the dataset
print("Available Record Sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"@id: {rs.id}, name: {rs.name}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    @id: {f.id}, name: {f.name}, dataType: {getattr(f, 'dataType', 'N/A')}")
    print()

To further understand the data, let's peek at a few sample records from a primary record set. 

**NOTE:** Replace `<record_set_id>` with a desired record set's `@id` from the previous cell. Here we use the first available record set for demonstration.

In [ ]:
# For demonstration, use the first record set found
selected_record_set = record_sets[0]
print(f"\nShowing 3 sample records from record set: {selected_record_set.id}")

for i, rec in enumerate(dataset.records(record_set=selected_record_set.id)):
    print(rec)
    if i >= 2: break

## 3. Data Extraction
Load data from record sets into Pandas DataFrames for further analysis.

We'll extract all primary tabular record sets. Replace or expand `record_set_ids` as appropriate for your analysis.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for record set {rs_id} (columns: {df.columns.tolist()})")

# Display head of first record set as example
first_rs_id = record_set_ids[0]
print(f"\nPreview of {first_rs_id}:")
display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping by attributes.

We'll select the first available numeric field from our main record set for demonstration. You can adjust `numeric_field_id` and `group_field_id` for more advanced analysis based on your overview above.

In [ ]:
# Assume first record set contains numeric fields, choose one for EDA
primary_rs = record_sets[0]
primary_rs_id = primary_rs.id

numeric_field_id = None
group_field_id = None

# Identify first numeric and group field by dataType, if possible
for field in getattr(primary_rs, 'fields', []):
    dt = getattr(field, 'dataType', '').lower()
    if numeric_field_id is None and (dt == 'integer' or dt == 'float'):
        numeric_field_id = field.id
    # Choose a group field (categorical)
    if group_field_id is None and (dt == 'text' or dt == 'string' or dt == 'boolean'):
        group_field_id = field.id

print(f"Selected numeric field: {numeric_field_id}")
print(f"Selected group field: {group_field_id}")

df = dataframes[primary_rs_id]
# If the dataset is empty, skip EDA
if not df.empty and numeric_field_id in df.columns:
    # Try to ensure numeric dtype
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Set a threshold based on data distribution (e.g., mean or a fixed number)
    threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (showing up to 5):")
    display(filtered_df.head())

    # Normalize the field (standard score)
    norm = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = norm
    print(f"\n{numeric_field_id} (normalized) (showing up to 5):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a selected categorical field, if present
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found or data is empty for EDA.")

## 5. Visualization
Visualize basic distributions or relationships between fields in the dataset.

We'll plot the distribution of the numeric field, and (if a group field is available) compare means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Grouped bar plot if group field is present
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=df, ci='sd')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No data to plot. Please check numeric_field_id and available data.")

## 6. Conclusion

In this notebook, we demonstrated how to:

- Load and review a FAIR² Croissant-based dataset with `mlcroissant`
- Inspect available record sets and fields by their `@id`
- Extract data into DataFrames and perform basic EDA (filtering, normalization, grouping)
- Visualize data distributions and group comparisons

This approach can be adapted to other Croissant datasets simply by adjusting the schema URL and the selected field `@id`s.

**Next steps:**
- Deepen analysis by exploring more fields or record sets
- Apply domain-specific data transformations or visualizations
- Use this workflow as a robust foundation for reproducible ML/data science pipelines using Croissant metadata!